# Trabajo Ev 1

## Leer DataSet


In [1]:
import pandas as pd
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

raw_df = pd.read_csv("retail_store_sales.csv")
#respaldo y copia del dataset original

df = raw_df
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


##Limpieza de | Price Per Unit | Quantity | Total Spent (Busqueda de Nulos)

In [2]:


#comprobamos la cantidad de campos nulos
df.isna().sum()


,0
Transaction ID,0
Customer ID,0
Category,0
Item,1213
Price Per Unit,609
Quantity,604
Total Spent,604
Payment Method,0
Location,0
Transaction Date,0


Podemos concluir que  TODOS los registros tienen un "TRANSACTION_ID", "CUSTOMER_ID", "CATEGORY","PAYMENT METHOD",
"LOCATION" y "TRANSACTION DATE".

In [3]:
#comprobamos si existen transacciones repetidas
id_repetidos= raw_df['Transaction ID'].duplicated().sum()
#comprobamos si en general existen  registros repetidos (aunque si no hay transacciones repetidas esto se puede evitar)
datos_Repetidos = raw_df.duplicated().sum()

print(f"la cantidad de transacciones repetidas es: {id_repetidos}")
print(f"la cantidad de datos repetidos es: {datos_Repetidos}")

la cantidad de transacciones repetidas es: 0
la cantidad de datos repetidos es: 0


no existen errores de duplicación y cada transacción es unica

In [4]:
# En los siguientes 4 codigos se buscarán patrones en los que no se pueden recuperar los valores de las columnas "Price Per Unit"	"Quantity"	"Total Spent"
# Esto debido a que no por ejemplo , no podemos sacar el total sin la cantidad ni el precio unitario
#en general si hay menos de 2 campos de estas 3 columnas, el dato es irrecuperable.
# Busqueda de si "Precio", "Cantidad", "Total" son Nulos de manera simultanea

#Casos donde 'Price per unit', 'Quantity' y 'Total spent' son nulos
filtro_null_1 = df.loc[
      df['Price Per Unit'].isnull() & df['Quantity'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_1.shape

(0, 3)

In [5]:
#Casos donde 'Cantidad' y 'Total' son Nulos
filtro_null_2 = df.loc[
      df['Quantity'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_2.shape

(604, 3)

In [6]:
#Casos donde 'Price per Unit y 'Quantity' son Nulos
filtro_null_3 = df.loc[
      df['Price Per Unit'].isnull() & df['Quantity'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_3.shape

(0, 3)

In [7]:

#Casos donde 'Price per Unit y 'Total' son Nulos
filtro_null_4 =df.loc[
      df['Price Per Unit'].isnull() & df['Total Spent'].isnull(),
      ['Price Per Unit', 'Quantity', 'Total Spent']
    ]
filtro_null_4.shape

(0, 3)

Podemos observar que de los 4 escenarios posibles donde NO se pueden recuperar estos 3 datos, solo hay  604 escenarios donde no hay cantidad, ni precio unitario, en el resto de casos es recuperable la información por ende, solo limpiamos ese escenario.

In [8]:
#borramos los escenarios de 'Price per unit', 'Quantity' y 'Total spent' donde no es recuperable
df.drop(filtro_null_2.index, axis=0, inplace=True)
#revisamos los nulls denuevo
df.isna().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,609
Price Per Unit,609
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


## Restauracion de | Price Per Unit | Quantity | Total Spent  (restaurar algunos campos vacios)

Podemos observar del bloque de codigo anterior que los unicos campos nulos actuales entre 'Price Per Unit', 'Quantity' y 'Total Spent' solo hay 609 casos nulos de Price Per unit


In [9]:

#Buscaremos reparar los campos vacios de 'Price Per unit' donde NO hay descuento (ya que el descuento nos puede dar un valor sesgado del total real)

#PRICE PER UNIT NULL

df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Total Spent'] / df['Quantity'])

df.isna().sum()



,0
Transaction ID,0
Customer ID,0
Category,0
Item,609
Price Per Unit,0
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


##Restauracion de |Items|

antes de intentar restaurar los Items, necesitamos explorar información que podria tener relevancia con items como su categoria y valores unicos

In [10]:
#Contamos la cantidad de categorias que existen
cat_size = df['Category'].value_counts()
print(cat_size)

Category
Furniture                             1525
Electric household essentials         1516
Milk Products                         1513
Food                                  1507
Butchers                              1496
Beverages                             1496
Computers and electric accessories    1477
Patisserie                            1441
Name: count, dtype: int64


In [11]:
#items unicos por cada categoria
cat_unique = df.groupby('Category')['Item'].nunique()
print(cat_unique)

Category
Beverages                             25
Butchers                              25
Computers and electric accessories    25
Electric household essentials         25
Food                                  25
Furniture                             25
Milk Products                         25
Patisserie                            25
Name: Item, dtype: int64


In [12]:
#items unicos
items_unique = df.loc[df['Item'].notna(), 'Item'].unique()
print(len(items_unique))

200


hay 200 items unicos y 8 categorias, por cada categoria hay 25 items unicos que no se repite.

Ahora vereficaremos si cada item tiene un unico valor unitario.

In [13]:
#verificaremos que cada item tenga un valor unitario unico (es de cir que un mismo item no tenga 2 valores unitario distintos)

#eliminamos los campos nulos de item para ahorrarnos problemas con el groupby y lo guardaremos en un DF temporal 'not_null_items'
not_null_items = df.loc[df['Item'].notna()].copy()

#hacemos un group by de items donde cada item guardara un array de todos los 'Price Per Unit' unicos que hay
item_prices_array = not_null_items.groupby('Item')['Price Per Unit'].unique()

#evaluamos en 'resultado' si es que existen Items cuyo array de 'Price Per Unit' tenga mas de 1 valor
resultado = item_prices_array[item_prices_array.apply(len)>1]
print(f'Existen {len(resultado)} items que poseen mas de 1 Price Per Unit')


Existen 0 items que poseen mas de 1 Price Per Unit


Por lo tanto podemos garantizar que cada Item tiene un solo 'Price Per Unit'.
Con esta información podriamos restaurar algunos items nulos si logramos identificar los items mediante el precio unitario y categoria **(siempre y cuando no exista mas de un item en una misma categoria que tenga el mismo precio unitario)**

In [26]:
#nuevamente re-utilizamos el DF "not_null_items" para indentificar y trabajar información relacionada con los Items conocidos.
#creamos un DF temporal agrupando los precios untarios por categoria e item. Ademas en vez de usar Unique usaremos .mean para evitar operar con arrays, no afecta el resultado pues
#previamente ya confirmamos que cada item tiene un precio unitario unico por lo tanto su promedio deberia ser igual a cualquier valor del registro de ese mismo item
item_identify_df = not_null_items.groupby(['Category', 'Item'])['Price Per Unit'].mean()
#reseteamos en index para que el objeto series se pueda operar por columnas
item_identify_df = item_identify_df.reset_index()

#recolectamos los datos duplicados  y los almacenamos en una mascara
duplicado = item_identify_df.duplicated(subset=['Category', 'Price Per Unit'], keep=False)

# Solo para asegurarnos revisamos un DF aplicando la mascara de puplicados, si hay datos que cumplen el criterio de que en una misma categoria se repite el precio unitario
#se deberian poder observar
casos_duplicados = item_identify_df[duplicado]

print("Items que comparten precio dentro de la misma categoría:")
print(casos_duplicados)



Items que comparten precio dentro de la misma categoría:
Empty DataFrame
Columns: [Category, Item, Price Per Unit]
Index: []


Como podemos observar, cada item dentro de su propia categoria tiene un precio unitario unico, por lo que podemos crear un diccionario para identificar items apartir de su precio unitario y categoria.

In [36]:
#creamos el directorio aplicando un filtro incluyendo indexación y transformandola en un diccionario.
item_dict = not_null_items.set_index(['Category', 'Price Per Unit'])['Item'].to_dict()

#definimos una función para recuperar un item apartir de categoria y precio unitario recorriendo el diccionario.
def buscar_item(categoria, precio, diccionario):

    key= diccionario.get(categoria, float(precio))
    item = diccionario.get(key, 'Item no identificado')
    if item == 'Item no identificado':
      return
    else:
      return item




In [43]:
#empezamos a recuperar y rellenar los campos vacios:
null_item_mask = df[df['Item'].isna()]

#creamos una lista con cada valor category y price per unit
categorias = null_item_mask['Category'].tolist()
precios = null_item_mask['Price Per Unit'].tolist()
#Recuperamos los nombres de los items ejecutando el diccionario y los guardamos en "nombres_recuperados"
nombres_recuperados = []
for i in range(len(categorias)):
    llave = (categorias[i], precios[i])

    # Buscamos en el diccionario
    # .get() nos devuelve el nombre si existe, o None si no
    resultado = item_dict.get(llave)

    # Agregamos el resultado a nuestra lista de "bolsa"
    nombres_recuperados.append(resultado)

# Rellenamos los datos nulos con los items recuperados del diccionario
df.loc[null_item_mask.index, 'Item'] = nombres_recuperados


In [44]:
df.isna().sum()

,0
Transaction ID,0
Customer ID,0
Category,0
Item,0
Price Per Unit,0
Quantity,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
5,TXN_7482416,CUST_09,Patisserie,NaN,20.0,10.0,200.0,Credit Card,Online,2023-11-30,NaN
11,TXN_5422631,CUST_09,Milk Products,NaN,6.5,8.0,52.0,Digital Wallet,In-store,2025-01-12,True
17,TXN_9634894,CUST_15,Milk Products,NaN,27.5,10.0,275.0,Digital Wallet,Online,2022-04-17,NaN
21,TXN_8685338,CUST_15,Milk Products,NaN,35.0,3.0,105.0,Credit Card,In-store,2023-10-29,NaN
32,TXN_1543244,CUST_20,Food,NaN,24.5,8.0,196.0,Credit Card,Online,2024-10-25,True
